# 22 — Despliegue: el modelo de 5 variables en la web
Punto **4** del informe. Un modelo solo sirve si alguien lo puede usar. Aquí se toma la versión de **5 variables** del
notebook 21 y se publica en la web del proyecto (`web/prediccion.html`): se escriben los datos de un proceso y el
navegador calcula la probabilidad de que acumule mora, **sin servidor y sin instalar nada**.

Cómo se logra:
1. Se elige el mejor modelo de 5 variables **entre los que se pueden exportar** a un archivo de datos: los de árboles
   (Bagging, Random Forest, Gradient Boosting). Un árbol es solo una lista de preguntas «¿variable ≤ umbral?»; se copian
   esas preguntas y las hojas a un JSON y la página las recorre en JavaScript.
2. Se guarda el modelo entrenado (`models/modelo_5var.joblib`) y su versión exportada (`web/modelo/modelo_5var.json`), con una
   copia idéntica en `despliegue/sitio/modelo/`: la carpeta que se publica en el servidor con Docker Compose.
3. **Control:** una reimplementación que lee **solo el JSON** debe dar exactamente la misma probabilidad que scikit-learn
   en todos los procesos de prueba. La página repite ese control con 20 de ellos.

In [1]:
%run -i modulos/comun.ipynb
%run -i modulos/modelado.ipynb
import math
import warnings
import joblib
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
datos = cargar_conjunto()
part = pd.read_csv(MODELO_DIR / "particion.csv")
assert (part[OBJETIVO].to_numpy() == datos[OBJETIVO].to_numpy()).all()
tr = part.index[part["particion"] == "entrenamiento"]
te = part.index[part["particion"] == "prueba"]
y_tr, y_te = datos.loc[tr, OBJETIVO], datos.loc[te, OBJETIVO]
hp = leer_hp()
TOP5 = json.loads((MODELO_DIR / "top5.json").read_text(encoding="utf-8"))["variables"]
versiones = pd.read_csv(MODELO_DIR / "comparacion_versiones.csv")
v5 = versiones[versiones["version"] == "5 más importantes"].set_index("modelo")
EXPORTABLES = ["Bagging", "Random Forest", "Gradient Boosting"]
ELEGIDO = v5.loc[EXPORTABLES, "roc_auc_cv"].idxmax()
display(v5[["roc_auc_cv", "roc_auc_prueba"]].sort_values("roc_auc_cv", ascending=False).round(3))
print("variables:", TOP5)
print("modelo desplegado:", ELEGIDO, "(mejor ROC-AUC de validación cruzada entre los exportables)")

,roc_auc_cv,roc_auc_prueba
modelo,,
AdaBoost,0.714,0.660
Gradient Boosting,0.712,0.663
Random Forest,0.705,0.662
Stacking,0.705,0.664
Bagging,0.704,0.658
Regresión logística,0.659,0.619
SVM,0.649,0.631


variables: ['ingresos_por_juzgado', 'presion_stock', 'stock_por_juzgado', 'materia_homologada', 'ingresos_totales']
modelo desplegado: Gradient Boosting (mejor ROC-AUC de validación cruzada entre los exportables)


## Entrenar y guardar
Se entrena con el 80 % de entrenamiento (el mismo con que se evaluó), para que las métricas de la prueba sigan siendo
las del modelo publicado.

In [2]:
final = modelo(ELEGIDO, TOP5, hp).fit(datos.loc[tr, TOP5], y_tr)
p_te = final.predict_proba(datos.loc[te, TOP5])[:, 1]
pred = (p_te >= 0.5).astype(int)
metricas_final = {"roc_auc": roc_auc_score(y_te, p_te), "exactitud": accuracy_score(y_te, pred),
                  "precision": precision_score(y_te, pred), "recall": recall_score(y_te, pred), "f1": f1_score(y_te, pred)}
print({k: round(v, 3) for k, v in metricas_final.items()})
assert abs(metricas_final["roc_auc"] - v5.loc[ELEGIDO, "roc_auc_prueba"]) < 1e-9, "no es el mismo modelo del notebook 21"
CARPETA_MODELOS = RAIZ / "models"
CARPETA_MODELOS.mkdir(exist_ok=True)
joblib.dump(final, CARPETA_MODELOS / "modelo_5var.joblib")

{'roc_auc': 0.663, 'exactitud': 0.589, 'precision': 0.568, 'recall': 0.68, 'f1': 0.619}


['C:\\Users\\barry\\maxi\\u\\analisis\\congestion-judicial-bolivia\\models\\modelo_5var.joblib']

## Exportar a JSON
El JSON lleva: qué datos pedir (las columnas originales que necesitan las 5 variables), cómo calcular las variables de
ingeniería, el preprocesamiento (mediana para faltantes, categorías del one-hot) y los árboles.

In [3]:
entradas = []
for c in TOP5:
    for dep in DEPENDENCIAS.get(c, [c]):
        if dep not in entradas:
            entradas.append(dep)
desc_entradas = []
for c in entradas:
    if c in CATEGORICAS:
        desc_entradas.append({"columna": c, "nombre": NOMBRES[c], "tipo": "categorica",
                              "opciones": sorted(datos.loc[tr, c].dropna().unique().tolist()),
                              "moda": datos.loc[tr, c].mode().iloc[0]})
    else:
        v = datos.loc[tr, c]
        desc_entradas.append({"columna": c, "nombre": NOMBRES[c], "tipo": "numerica", "min": float(v.min()),
                              "mediana": float(v.median()), "p90": float(v.quantile(0.9)), "max": float(v.max())})

pre = final.named_steps["pre"]
assert [t[0] for t in pre.transformers_ if t[0] != "remainder"][:1] == ["num"]
num_cols = list(pre.transformers_[0][2])
num_pipe = pre.named_transformers_["num"]
pre_json = {"numericas": num_cols, "medianas": num_pipe.named_steps["imputar"].statistics_.tolist(),
            "escala": None, "categoricas": [], "categorias": []}
assert "escalar" not in num_pipe.named_steps, "los modelos de árboles no usan escala"
if "cat" in pre.named_transformers_:
    pre_json["categoricas"] = list(pre.transformers_[1][2])
    pre_json["categorias"] = [c.tolist() for c in pre.named_transformers_["cat"].named_steps["onehot"].categories_]


def arbol_json(arbol, regresion):
    t = arbol.tree_
    if regresion:
        hoja = t.value[:, 0, 0]
    else:
        hoja = t.value[:, 0, 1] / t.value[:, 0, :].sum(axis=1)
    return {"izq": t.children_left.tolist(), "der": t.children_right.tolist(), "var": t.feature.tolist(),
            "umbral": t.threshold.tolist(), "valor": hoja.tolist()}


est = final.named_steps["modelo"]
Xt_te = pre.transform(datos.loc[te, TOP5])
if ELEGIDO == "Random Forest":
    modelo_json = {"tipo": "promedio_arboles", "arboles": [arbol_json(a, False) for a in est.estimators_],
                   "subconjuntos": None}
elif ELEGIDO == "Bagging":
    modelo_json = {"tipo": "promedio_arboles", "arboles": [arbol_json(a, False) for a in est.estimators_],
                   "subconjuntos": [s.tolist() for s in est.estimators_features_]}
else:
    suma = sum(a[0].predict(Xt_te.astype(np.float32)) for a in est.estimators_)
    inicial = est.decision_function(Xt_te) - est.learning_rate * suma
    assert np.ptp(inicial) < 1e-9, "el valor inicial del boosting debería ser constante"
    modelo_json = {"tipo": "boosting", "inicial": float(inicial[0]), "tasa_aprendizaje": est.learning_rate,
                   "arboles": [arbol_json(a[0], True) for a in est.estimators_]}

prueba = datos.loc[te].sample(20, random_state=SEMILLA)
autoverificacion = [{"entrada": {c: (r[c] if c in CATEGORICAS else float(r[c])) for c in entradas},
                     "probabilidad": float(final.predict_proba(prueba.loc[[i], TOP5])[0, 1]),
                     "real": int(r[OBJETIVO]), "proceso": r["tipo_proceso"], "territorio": r["territorio"],
                     "materia": r["materia_homologada"]}
                    for i, r in prueba.iterrows()]
v27 = versiones[versiones["version"] == "con ingeniería (27)"].set_index("modelo")
export = {
    "descripcion": "Probabilidad de que un proceso no penal acumule mora (resuelva menos causas de las que ingresan)",
    "modelo": ELEGIDO, "hiperparametros": hp[ELEGIDO], "variables": TOP5,
    "nombres": {c: NOMBRES[c] for c in TOP5 + entradas}, "formulas": {c: FORMULAS[c] for c in TOP5 if c in FORMULAS},
    "entradas": desc_entradas, "preprocesamiento": pre_json, "umbral": 0.5,
    "metricas_prueba": metricas_final,
    "comparacion": {"roc_auc_prueba_27_variables": float(v27.loc[ELEGIDO, "roc_auc_prueba"]),
                    "roc_auc_cv_5_variables": float(v5.loc[ELEGIDO, "roc_auc_cv"])},
    "n_entrenamiento": int(len(tr)), "n_prueba": int(len(te)), "autoverificacion": autoverificacion,
    "arbol_compara_float32": True, **modelo_json,
}
(RAIZ / "web" / "modelo").mkdir(exist_ok=True)
ruta_json = RAIZ / "web" / "modelo" / "modelo_5var.json"
ruta_json.write_text(json.dumps(export, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
# La misma copia va a la carpeta del despliegue en el servidor (despliegue/sitio), que se publica con Docker.
ruta_despliegue = RAIZ / "despliegue" / "sitio" / "modelo" / "modelo_5var.json"
ruta_despliegue.parent.mkdir(parents=True, exist_ok=True)
ruta_despliegue.write_bytes(ruta_json.read_bytes())
assert ruta_despliegue.read_bytes() == ruta_json.read_bytes()
print(f"{ruta_json.name}: {ruta_json.stat().st_size / 1e6:.2f} MB, {len(modelo_json['arboles'])} árboles; entradas: {entradas}")

modelo_5var.json: 0.19 MB, 200 árboles; entradas: ['ingresos_totales', 'num_juzgados_pagina', 'pendientes_inicio_efectivo', 'materia_homologada']


## Control: predecir leyendo solo el JSON
Esta función no usa scikit-learn: hace a mano lo mismo que hará la página (calcular las variables nuevas, imputar,
one-hot, recorrer los árboles). Los árboles de scikit-learn comparan los datos convertidos a `float32`; la reimplementación
también, o un dato justo en el umbral podría ir por la rama equivocada.

In [4]:
J = json.loads(ruta_json.read_text(encoding="utf-8"))


def variables_desde_entradas(e):
    s, i, j = e.get("pendientes_inicio_efectivo"), e.get("ingresos_totales"), e.get("num_juzgados_pagina")
    calc = {
        "log_stock_inicial": lambda: math.log1p(s),
        "log_ingresos": lambda: math.log1p(i),
        "presion_stock": lambda: s / (i + 1),
        "ingresos_por_juzgado": lambda: i / j,
        "stock_por_juzgado": lambda: s / j,
        "prop_acefalias": lambda: e["personal_items_acefalias"] / e["personal_items_total"],
        "prop_excusa": lambda: e["recibidas_excusa_recusacion"] / (i + 1),
        "sin_stock_inicial": lambda: 1.0 if s == 0 else 0.0,
    }
    return {c: (calc[c]() if c in calc else e[c]) for c in J["variables"]}


def recorrer(arbol, x):
    n = 0
    while arbol["izq"][n] != -1:
        n = arbol["izq"][n] if float(np.float32(x[arbol["var"][n]])) <= arbol["umbral"][n] else arbol["der"][n]
    return arbol["valor"][n]


def predecir_json(e):
    v = variables_desde_entradas(e)
    p = J["preprocesamiento"]
    x = [v[c] if v[c] is not None and not (isinstance(v[c], float) and math.isnan(v[c])) else med
         for c, med in zip(p["numericas"], p["medianas"])]
    for c, cats in zip(p["categoricas"], p["categorias"]):
        x += [1.0 if v[c] == k else 0.0 for k in cats]
    if J["tipo"] == "promedio_arboles":
        subs = J["subconjuntos"] or [None] * len(J["arboles"])
        return sum(recorrer(a, x if s is None else [x[k] for k in s]) for a, s in zip(J["arboles"], subs)) / len(J["arboles"])
    raw = J["inicial"] + J["tasa_aprendizaje"] * sum(recorrer(a, x) for a in J["arboles"])
    return 1 / (1 + math.exp(-raw))


p_json = np.array([predecir_json({c: datos.loc[k, c] for c in entradas}) for k in te])
dif = np.abs(p_json - p_te).max()
print(f"diferencia máxima JSON vs scikit-learn en {len(te)} procesos de prueba: {dif:.2e}")
assert dif < 1e-9
assert all(abs(predecir_json(a["entrada"]) - a["probabilidad"]) < 1e-9 for a in J["autoverificacion"])

diferencia máxima JSON vs scikit-learn en 248 procesos de prueba: 3.33e-16


## Tablas y cifras para el informe LaTeX
Se escriben desde los CSV de los notebooks 19 a 22 a `informe/tablas/`, para que el informe no copie cifras a mano.

In [5]:
TABLAS = RAIZ / "informe" / "tablas"
TABLAS.mkdir(parents=True, exist_ok=True)


def tex(s):
    s = str(s)
    for a, b in [("\\", r"\textbackslash{}"), ("_", r"\_"), ("%", r"\%"), ("&", r"\&"), ("#", r"\#"), ("$", r"\$")]:
        s = s.replace(a, b)
    return s


def num(x, d=3):
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and float(x).is_integer() and d == 0):
        return f"{int(x):,}".replace(",", ".")
    return f"{x:.{d}f}".replace(".", ",")


def tabla(nombre, encabezado, filas, columnas):
    cuerpo = "\n".join(" & ".join(f) + r" \\" for f in filas)
    (TABLAS / f"{nombre}.tex").write_text(
        f"\\begin{{tabular}}{{{columnas}}}\n\\toprule\n{' & '.join(encabezado)} \\\\\n\\midrule\n{cuerpo}\n\\bottomrule\n\\end{{tabular}}\n",
        encoding="utf-8")


variables = pd.read_csv(MODELO_DIR / "variables.csv").fillna("")
corr = pd.read_csv(MODELO_DIR / "eda_correlaciones.csv").set_index("variable")
mod = variables[variables["grupo"] != "excluida (fuga)"]
tabla("variables", ["Variable", "Descripción", "Grupo", "Conversión", r"$\rho$ Spearman"],
      [[r"\texttt{" + tex(r.variable) + "}", tex(r.nombre if not r.formula else r.formula), tex(r.grupo),
        tex(r.conversion.replace(" + escala (SVM)", "").replace("imputación mediana", "mediana")),
        num(corr.loc[r.variable, "spearman"], 2) if r.variable in corr.index else "---"]
       for r in mod.itertuples()], "llllr")
fuga = variables[variables["grupo"] == "excluida (fuga)"]
tabla("fuga", ["Columna excluida", "Motivo"],
      [[r"\texttt{" + tex(r.variable) + "}", tex(r.motivo_exclusion)] for r in fuga.itertuples()], "lp{9cm}")

mcv = pd.read_csv(MODELO_DIR / "metricas_cv.csv")
pv = mcv.pivot_table(index="modelo", columns="configuracion", values="roc_auc").reindex(MODELOS)
aj = mcv[mcv["configuracion"] == "ajustado"].set_index("modelo").reindex(MODELOS)
tabla("metricas_cv", ["Modelo", "Familia", "AUC por defecto", "AUC ajustado", "Desvío", "Exactitud", "F1"],
      [[tex(m), tex(FAMILIAS[m][0]), num(pv.loc[m, "por defecto"]), num(pv.loc[m, "ajustado"]),
        num(aj.loc[m, "roc_auc_desvio"]), num(aj.loc[m, "exactitud"]), num(aj.loc[m, "f1"])] for m in MODELOS], "llrrrrr")

def valor_hp(v):
    """Valor legible: None = sin límite, enteros sin decimales, decimales con 3 cifras significativas."""
    if v is None:
        return "sin límite"
    if isinstance(v, bool) or isinstance(v, str):
        return tex(v)
    if float(v).is_integer():
        return num(int(v), 0)
    return f"{v:.3g}".replace(".", ",")


tabla("hiperparametros", ["Modelo", "Hiperparámetro", "Valor elegido"],
      [[tex(m), r"\texttt{" + tex(k) + "}", valor_hp(v)] for m, p in hp.items() for k, v in p.items()], "lll")

mt = pd.read_csv(MODELO_DIR / "metricas_test.csv").set_index("modelo").reindex(MODELOS)
tabla("metricas_test", ["Modelo", "Exactitud", "Precisión", "Recall", "F1", "ROC-AUC"],
      [[tex(m), num(r.exactitud), num(r.precision), num(r.recall), num(r.f1), num(r.roc_auc)] for m, r in mt.iterrows()],
      "lrrrrr")

imp = pd.read_csv(MODELO_DIR / "importancia.csv")
tabla("importancia", ["\\#", "Variable", "Grupo", "Permutación", "Desvío", "Impureza RF"],
      [[str(r.rango), tex(r.nombre), tex(r.grupo), num(r.importancia, 4), num(r.desvio_entre_pliegues, 4),
        num(r.importancia_impureza_rf, 3)] for r in imp.head(12).itertuples()], "rllrrr")

ver = pd.read_csv(MODELO_DIR / "comparacion_versiones.csv")
orden_v = ["originales (19)", "con ingeniería (27)", "5 más importantes"]
pcv = ver.pivot_table(index="modelo", columns="version", values="roc_auc_cv")
pte = ver.pivot_table(index="modelo", columns="version", values="roc_auc_prueba")
mods = [m for m in MODELOS if m in pcv.index]
tabla("versiones", ["Modelo", "19 (CV)", "19 (prueba)", "27 (CV)", "27 (prueba)", "5 (CV)", "5 (prueba)"],
      [[tex(m)] + [num(x) for v in orden_v for x in (pcv.loc[m, v], pte.loc[m, v])] for m in mods], "lrrrrrr")

rob = pd.read_csv(MODELO_DIR / "robustez_grupos.csv")
tabla("robustez", ["Versión", "Modelo", "AUC CV normal", "AUC por grupos", "Diferencia"],
      [[tex(r.version), tex(r.modelo), num(r.roc_auc_cv_normal), num(r.roc_auc_por_grupos), num(r.diferencia)]
       for r in rob.itertuples()], "llrrr")
print(sorted(p.name for p in TABLAS.glob("*.tex")))

['cifras.tex', 'fuga.tex', 'hiperparametros.tex', 'importancia.tex', 'metricas_cv.tex', 'metricas_test.tex', 'robustez.tex', 'variables.tex', 'versiones.tex']


In [6]:
mejor_cv = aj["roc_auc"].drop("Línea base (clase mayoritaria)").idxmax()
curado_modelo = pd.read_parquet(CURATED / "features" / "dataset_analitico_curado.parquet")
curado_modelo = curado_modelo[(curado_modelo["tipo_elemento_analitico"] == "proceso") & (curado_modelo["atendidas"] > 0)]
assert len(curado_modelo) == len(datos)
cm = pd.read_csv(MODELO_DIR / "matriz_confusion.csv", index_col=0)
y_all = datos[OBJETIVO]
macros = {
    "nFilas": num(len(datos), 0), "nEntrenamiento": num(len(tr), 0), "nPrueba": num(len(te), 0),
    "nAcumulan": num(int(y_all.sum()), 0), "pctAcumulan": num(100 * y_all.mean(), 1),
    "nTiposProceso": num(datos["tipo_proceso"].nunique(), 0),
    "nOriginales": str(len(ORIGINALES)), "nIngenieria": str(len(INGENIERIA)), "nCompleta": str(len(COMPLETA)),
    "nFuga": str(len(FUGA)),
    "mejorModelo": tex(mejor_cv), "mejorAucCv": num(aj.loc[mejor_cv, "roc_auc"]),
    "mejorAucPrueba": num(mt.loc[mejor_cv, "roc_auc"]), "mejorExactitudPrueba": num(mt.loc[mejor_cv, "exactitud"]),
    "mejorRecallPrueba": num(mt.loc[mejor_cv, "recall"]), "mejorPrecisionPrueba": num(mt.loc[mejor_cv, "precision"]),
    "cmVN": str(cm.iloc[0, 0]), "cmFP": str(cm.iloc[0, 1]), "cmFN": str(cm.iloc[1, 0]), "cmVP": str(cm.iloc[1, 1]),
    "modeloDesplegado": tex(ELEGIDO), "aucDesplegado": num(metricas_final["roc_auc"]),
    "exactitudDesplegado": num(metricas_final["exactitud"]), "recallDesplegado": num(metricas_final["recall"]),
    "precisionDesplegado": num(metricas_final["precision"]),
    "aucDesplegadoVeintisiete": num(export["comparacion"]["roc_auc_prueba_27_variables"]),
    "nArboles": str(len(modelo_json["arboles"])), "pesoJson": num(ruta_json.stat().st_size / 1e6, 1),
    "difJson": "0" if dif == 0 else f"{dif:.0e}".replace("e-", r"\times10^{-") + "}",
}
for k, c in enumerate(TOP5, 1):
    macros[f"topVar{'ABCDE'[k - 1]}"] = tex(NOMBRES[c])

# Una macro por modelo y por versión, para que el texto del informe no copie cifras a mano.
CLAVE_MODELO = {"Regresión logística": "Logistica", "SVM": "SVM", "Bagging": "Bagging", "Random Forest": "RandomForest",
                "AdaBoost": "AdaBoost", "Gradient Boosting": "GradientBoosting", "Stacking": "Stacking"}
CLAVE_VERSION = {"originales (19)": "Diecinueve", "con ingeniería (27)": "Veintisiete", "5 más importantes": "Cinco"}
for m, k in CLAVE_MODELO.items():
    macros[f"aucDef{k}"] = num(pv.loc[m, "por defecto"])
    macros[f"aucCv{k}"] = num(aj.loc[m, "roc_auc"])
    macros[f"aucPrueba{k}"] = num(mt.loc[m, "roc_auc"])
    macros[f"exactitudPrueba{k}"] = num(mt.loc[m, "exactitud"])
    for v, kv in CLAVE_VERSION.items():
        macros[f"aucV{kv}Cv{k}"] = num(pcv.loc[m, v])
        macros[f"aucV{kv}Prueba{k}"] = num(pte.loc[m, v])
for r in rob.itertuples():
    macros[f"aucGrupos{CLAVE_VERSION[r.version]}{CLAVE_MODELO[r.modelo]}"] = num(r.roc_auc_por_grupos)
    macros[f"difGrupos{CLAVE_VERSION[r.version]}{CLAVE_MODELO[r.modelo]}"] = num(r.diferencia)
sin_tasa = curado_modelo["tasa_congestion"].isna()
macros |= {
    "nSinTasa": num(int(sin_tasa.sum()), 0), "nSinTasaAcumulan": num(int(datos.loc[sin_tasa.to_numpy(), OBJETIVO].sum()), 0),
    "svmC": num(hp["SVM"]["C"], 0), "svmGamma": num(hp["SVM"]["gamma"], 3),
    "aucSvmMin": num(pd.read_csv(MODELO_DIR / "grilla_svm.csv", index_col=0).values.min()),
    "nEntradas": str(len(entradas)),
    "rhoIngresos": num(corr.loc["ingresos_totales", "spearman"], 2), "rhoPresion": num(corr.loc["presion_stock", "spearman"], 2),
    "rhoStock": num(corr.loc["pendientes_inicio_efectivo", "spearman"], 2),
}
lineas = [f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in macros.items()]
(TABLAS / "cifras.tex").write_text("% Generado por notebooks/22_despliegue.ipynb\n" + "\n".join(lineas) + "\n", encoding="utf-8")
print("\n".join(lineas))

\newcommand{\nFilas}{1.240}
\newcommand{\nEntrenamiento}{992}
\newcommand{\nPrueba}{248}
\newcommand{\nAcumulan}{608}
\newcommand{\pctAcumulan}{49,0}
\newcommand{\nTiposProceso}{87}
\newcommand{\nOriginales}{19}
\newcommand{\nIngenieria}{8}
\newcommand{\nCompleta}{27}
\newcommand{\nFuga}{18}
\newcommand{\mejorModelo}{Bagging}
\newcommand{\mejorAucCv}{0,753}
\newcommand{\mejorAucPrueba}{0,729}
\newcommand{\mejorExactitudPrueba}{0,661}
\newcommand{\mejorRecallPrueba}{0,697}
\newcommand{\mejorPrecisionPrueba}{0,644}
\newcommand{\cmVN}{79}
\newcommand{\cmFP}{47}
\newcommand{\cmFN}{37}
\newcommand{\cmVP}{85}
\newcommand{\modeloDesplegado}{Gradient Boosting}
\newcommand{\aucDesplegado}{0,663}
\newcommand{\exactitudDesplegado}{0,589}
\newcommand{\recallDesplegado}{0,680}
\newcommand{\precisionDesplegado}{0,568}
\newcommand{\aucDesplegadoVeintisiete}{0,736}
\newcommand{\nArboles}{200}
\newcommand{\pesoJson}{0,2}
\newcommand{\difJson}{3\times10^{-16}}
\newcommand{\topVarA}{ingresos por juzgado}

## Lectura
- **Modelo publicado: Gradient Boosting con 5 variables** (200 árboles en secuencia). Entre los exportables fue el de mejor
  validación cruzada con 5 variables (0,712; Random Forest 0,705, Bagging 0,704). AdaBoost tuvo 0,714, pero está fuera de los
  exportables y la diferencia está dentro del ruido.
- **Para usarlo basta con 4 datos:** ingresos del año, juzgados de esa materia en el territorio, stock heredado y materia.
  Las otras tres variables (ingresos por juzgado, presión del stock, stock por juzgado) se calculan con las mismas fórmulas.
- **En la prueba:** ROC-AUC 0,663, exactitud 0,589, recall 0,680 (detecta 2 de cada 3 procesos que acumulan) y precisión
  0,568. El mismo modelo con las 27 variables daba 0,736: **la versión de 5 variables es la que pidió el informe, y el costo
  de simplificar queda medido**. Sirve para ordenar qué procesos mirar primero, no para decidir sobre un juzgado.
- **El JSON es fiel:** la reimplementación sin scikit-learn coincide en los 248 procesos de prueba con una diferencia máxima
  de 3·10⁻¹⁶ (el redondeo de la última cifra de un número de máquina). La página repite el control con 20 procesos.
- **Las tablas y cifras del informe** quedan en `informe/tablas/`: el `.tex` las incluye con `\input`, así que si se vuelve a
  correr el pipeline, el informe se actualiza solo al compilar.
